In [1]:
import glob
import os
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
from collections import defaultdict
from sklearn.model_selection import train_test_split



In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
base_path = "/content/drive/MyDrive/dev_phase/subtask1/train/"
files = glob.glob(os.path.join(base_path, "*.csv"))

data = {}

for file in files:
    lang = os.path.splitext(os.path.basename(file))[0]  # amh, arb, eng

    df = pd.read_csv(file)

    data[lang] = {
        "X": df["text"].tolist(),
        "y": df["polarization"].tolist(),
        "df": df
    }

print("Loaded languages:", sorted(data.keys()))


Loaded languages: ['amh', 'arb', 'ben', 'deu', 'eng', 'fas', 'hau', 'hin', 'ita', 'khm', 'mya', 'nep', 'ori', 'pan', 'pol', 'rus', 'spa', 'swa', 'tel', 'tur', 'urd', 'zho']


In [5]:
model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
embedding_model = AutoModel.from_pretrained(model_name).to(device)
embedding_model.eval()

for p in embedding_model.parameters():
    p.requires_grad = False

def mean_pooling(model_output, attention_mask):
    token_embeds = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeds.size()).float()
    sum_embeddings = torch.sum(token_embeds * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

In [6]:
def get_all_embeddings(texts, model, tokenizer, device, batch_size=32):
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(batch_texts, padding=True, truncation=True, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = model(**enc)
            embeddings = mean_pooling(outputs, enc['attention_mask'])
        all_embs.append(embeddings.cpu())
    return torch.cat(all_embs, dim=0)


embeddings_by_lang = {}

for lang, content in data.items():   # data[lang] has "X" and "y"
    print(f"Embedding language: {lang}")

    X_text = content["X"]
    y_labels = content["y"]

    X_emb = get_all_embeddings(X_text, embedding_model, tokenizer, device)
    y_tensor = torch.tensor(y_labels, dtype=torch.long)

    embeddings_by_lang[lang] = {
        "X": X_emb,
        "y": y_tensor
    }


Embedding language: urd


Embedding: 100%|██████████| 112/112 [00:33<00:00,  3.35it/s]


Embedding language: fas


Embedding: 100%|██████████| 103/103 [00:32<00:00,  3.19it/s]


Embedding language: ori


...

Embedding: 100%|██████████| 105/105 [00:50<00:00,  2.06it/s]


Embedding language: eng


Embedding: 100%|██████████| 101/101 [00:28<00:00,  3.57it/s]


Embedding language: mya


Embedding: 100%|██████████| 91/91 [01:04<00:00,  1.42it/s]


In [7]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):
        # x: (B, 1, D)
        out, _ = self.lstm(x)
        out = out[:, -1, :]   # last timestep
        return self.fc(out)


In [8]:
def train_lstm_cv_stats(X, y, device, num_classes,
                        k=5, epochs=10, batch_size=32, lr=1e-3):

    X = X.unsqueeze(1)  # (N,1,D)

    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    accs = []
    f1s  = []

    for fold, (tr, te) in enumerate(skf.split(X, y)):
        print(f"\nFold {fold+1}")

        X_tr, X_te = X[tr], X[te]
        y_tr, y_te = y[tr], y[te]

        train_ds = torch.utils.data.TensorDataset(X_tr, y_tr)
        test_ds  = torch.utils.data.TensorDataset(X_te, y_te)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        test_loader  = DataLoader(test_ds, batch_size=batch_size)

        model = LSTMClassifier(
            input_dim=X.shape[-1],
            hidden_dim=256,
            num_classes=num_classes
        ).to(device)

        opt = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()

        # ---- train ----
        for ep in range(epochs):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)

                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                opt.step()

        # ---- eval ----
        model.eval()
        preds = []

        with torch.no_grad():
            for xb, _ in test_loader:
                xb = xb.to(device)
                out = model(xb)
                preds.extend(out.argmax(1).cpu().numpy())

        acc = accuracy_score(y_te, preds)
        f1  = f1_score(y_te, preds, average="macro")

        print(f"ACC={acc:.4f} | F1={f1:.4f}")

        accs.append(acc)
        f1s.append(f1)

    return {
        "mean_acc": np.mean(accs),
        "std_acc":  np.std(accs),
        "mean_f1":  np.mean(f1s),
        "std_f1":   np.std(f1s)
    }


In [9]:
results = []

for lang in embeddings_by_lang:

    print(f"\n######## {lang.upper()} ########")

    X = embeddings_by_lang[lang]["X"]
    y = embeddings_by_lang[lang]["y"]

    num_classes = len(torch.unique(y))

    stats = train_lstm_cv_stats(
        X, y,
        device=device,
        num_classes=num_classes,
        k=5,
        epochs=10,
        batch_size=32
    )

    results.append({
        "language": lang,
        "mean_acc": stats["mean_acc"],
        "std_acc": stats["std_acc"],
        "mean_macro_f1": stats["mean_f1"],
        "std_macro_f1": stats["std_f1"]
    })

final_df = pd.DataFrame(results)
final_df = final_df.sort_values("mean_macro_f1", ascending=False)

final_df



######## URD ########

Fold 1
ACC=0.7616 | F1=0.7260

Fold 2
ACC=0.7546 | F1=0.6368

Fold 3
ACC=0.8036 | F1=0.7711

Fold 4
ACC=0.7978 | F1=0.7466

Fold 5
ACC=0.7823 | F1=0.7175

######## FAS ########

Fold 1
ACC=0.8361 | F1=0.7971

Fold 2
ACC=0.8346 | F1=0.8000

Fold 3
ACC=0.8452 | F1=0.7895

Fold 4
ACC=0.8376 | F1=0.7797

Fold 5
ACC=0.8422 | F1=0.7816

######## ORI ########

Fold 1
ACC=0.8122 | F1=0.7767

Fold 2
ACC=0.8059 | F1=0.7550

Fold 3
ACC=0.8101 | F1=0.7555

Fold 4
ACC=0.8013 | F1=0.7780

Fold 5
ACC=0.8372 | F1=0.8124

######## ARB ########

Fold 1
ACC=0.8003 | F1=0.8000

Fold 2
ACC=0.7944 | F1=0.7927

Fold 3
ACC=0.7885 | F1=0.7849

Fold 4
ACC=0.7825 | F1=0.7748

Fold 5
ACC=0.7973 | F1=0.7942

######## POL ########

Fold 1
ACC=0.7662 | F1=0.7466

Fold 2
ACC=0.7950 | F1=0.7907

Fold 3
ACC=0.7636 | F1=0.7599

Fold 4
ACC=0.7427 | F1=0.7257

Fold 5
ACC=0.7448 | F1=0.7068

######## AMH ########

Fold 1
ACC=0.8156 | F1=0.7036

Fold 2
ACC=0.7976 | F1=0.7062

Fold 3
ACC=0.8063 | F1=0

,language,mean_acc,std_acc,mean_macro_f1,std_macro_f1
7,tel,0.868546,0.019762,0.867780,0.020643
6,zho,0.850701,0.010511,0.850487,0.010445
10,nep,0.849377,0.014334,0.848112,0.015509
21,mya,0.819316,0.021889,0.811005,0.020636
11,ben,0.802888,0.025925,0.799563,0.025019
1,fas,0.839150,0.003957,0.789592,0.008084
3,arb,0.792604,0.006373,0.789339,0.008708
2,ori,0.813350,0.012508,0.775552,0.020916
20,eng,0.786774,0.023164,0.756393,0.030484
17,swa,0.757120,0.013651,0.755603,0.015703


from matplotlib import pyplot as plt
_df_0['index'].plot(kind='hist', bins=20, title='index')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_1['mean_acc'].plot(kind='hist', bins=20, title='mean_acc')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_2['std_acc'].plot(kind='hist', bins=20, title='std_acc')
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
_df_7.plot(kind='scatter', x='mean_macro_f1', y='std_macro_f1', s=32, alpha=.8)
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_8['index'].plot(kind='line', figsize=(8, 4), title='index')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_9['mean_acc'].plot(kind='line', figsize=(8, 4), title='mean_acc')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_10['std_acc'].plot(kind='line', figsize=(8, 4), title='std_acc')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_11['mean_macro_f1'].plot(kind='line', figsize=(8, 4), title='mean_macro_f1')
plt.gca().spines[['top', 'right']].set_visible(False)

In [10]:
final_df.to_csv("e5_lstm.csv", index=False)


In [11]:
print(final_df)

   language  mean_acc   std_acc  mean_macro_f1  std_macro_f1
7       tel  0.868546  0.019762       0.867780      0.020643
6       zho  0.850701  0.010511       0.850487      0.010445
10      nep  0.849377  0.014334       0.848112      0.015509
21      mya  0.819316  0.021889       0.811005      0.020636
11      ben  0.802888  0.025925       0.799563      0.025019
1       fas  0.839150  0.003957       0.789592      0.008084
3       arb  0.792604  0.006373       0.789339      0.008708
2       ori  0.813350  0.012508       0.775552      0.020916
20      eng  0.786774  0.023164       0.756393      0.030484
17      swa  0.757120  0.013651       0.755603      0.015703
9       hin  0.886658  0.010997       0.752562      0.030534
4       pol  0.762441  0.018851       0.745963      0.028796
12      tur  0.741124  0.018652       0.737508      0.021520
14      pan  0.739412  0.020764       0.736934      0.022554
18      spa  0.732829  0.008264       0.732160      0.008056
13      rus  0.776881  0